# PUEEA — Programa para el Uso Eficiente y Ahorro del Agua

**Bloque:** A — Gestión | **Línea temática:** `pueea` | **Variable principal:** consumo de agua (m³/mes)

---

## ¿Qué es el PUEEA?

El **Programa para el Uso Eficiente y Ahorro del Agua (PUEEA)** es el instrumento de gestión hídrica establecido por la **Ley 373 de 1997**. Su objetivo es garantizar la sostenibilidad del recurso hídrico mediante la reducción de pérdidas, la adopción de tecnologías de bajo consumo, la reutilización del agua y el control de captaciones ilegales.

El PUEEA es **obligatorio** para todos los usuarios que requieren concesión de aguas, incluyendo acueductos municipales, empresas industriales, proyectos agroindustriales y riego. Los usuarios con concesiones menores pueden acogerse al **PUEAA Simplificado** (Resolución 1257/2018). Todos los reportes deben cargarse mensualmente en el **SIRH (Sistema de Información del Recurso Hídrico)** del IDEAM.

## Meta de eficiencia hídrica

La meta operativa del IDEAM para usuarios grandes del PUEEA es una **reducción del 15% del consumo** respecto a la línea base medida en los primeros dos años del programa. El análisis estadístico de esta línea permite:
- Cuantificar si el consumo histórico muestra la tendencia requerida.
- Proyectar el consumo futuro bajo escenarios de eficiencia.
- Identificar meses o períodos de consumo anómalo que justifiquen inspección técnica.

## Marco normativo PUEEA

| Norma | Contenido clave |
|---|---|
| **Ley 373/1997** | Crea el PUEEA; obligatorio para usuarios con concesión |
| **Decreto 1076/2015** | Compilación ambiental; reglamenta concesiones y manejo del recurso |
| **Decreto 1090/2018** | Reglamenta técnicamente el PUEAA (métricas, formatos) |
| **Resolución 1257/2018** | Estructura del PUEAA estándar y PUEAA Simplificado |
| **Decreto 303/2012** | Reporte mensual obligatorio al RURH-SIRH |
| **Resolución 943/2021 (CRA)** | Fórmulas de costo para acueductos; metas de desincentivo al consumo excesivo |

## Instituciones clave

| Institución | Rol en el PUEEA |
|---|---|
| **MADS** | Política nacional, Política Nacional de Gestión Integral del Recurso Hídrico (PNGIRH) |
| **IDEAM** | SIRH, ENA (Estudio Nacional del Agua), red hidrometeorológica |
| **CARs** | Control, otorgamiento de concesiones, aprobación de PUEAA |
| **CRA** | Regulación de acueductos, incentivos y desincentivos tarifarios |
| **DANE** | Estadísticas de consumo per cápita y cobertura de acueducto |

## Indicadores oficiales IDEAM/MADS

| Indicador | Sigla | Referencia |
|---|---|---|
| Índice de Agua No Contabilizada | IANC | Pérdidas: meta < 30% |
| Índice de Vulnerabilidad Hídrica | IVH | Riesgo de desabastecimiento |
| Índice de Uso del Agua | IUA | Demanda / Oferta disponible |
| Índice de Retención y Regulación Hídrica | IRH | Capacidad de regulación de la cuenca |

## Preguntas que responde este notebook

1. ¿Cuál es la tendencia histórica del consumo de agua del usuario analizado?
2. ¿Se ha alcanzado o se está en camino de alcanzar la meta de reducción del 15%?
3. ¿Existen meses o períodos de consumo anómalo que sugieran fugas, errores de medición o captaciones no autorizadas?
4. ¿Cuántos períodos el consumo superó la línea base del PUEEA? (análisis de excedencias)
5. ¿Qué modelo —SARIMA, SARIMAX, Prophet o XGBoost— produce el pronóstico de consumo más preciso para planificar la disponibilidad hídrica?

> **Ficha de dominio completa:** [`docs/fuentes/pueea.md`](../../docs/fuentes/pueea.md)  
> **Decisiones metodológicas:** [`docs/decisiones.md`](../../docs/decisiones.md)

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.features.climate import load_oni, enso_lagged
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "pueea"
VARIABLE = "consumo_agua"
UNIDAD = "m³"

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `pueea` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "pueea.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos

### Fuentes de datos para consumo de agua en Colombia

Los datos de consumo de agua para el PUEEA provienen de registros administrativos de concesiones y de mediciones directas de caudal:

| Fuente | Variable | Frecuencia | Acceso |
|---|---|---|---|
| **SIRH (IDEAM)** | Volúmenes de captación por concesión | Mensual | [sirh.ideam.gov.co](http://sirh.ideam.gov.co) |
| **RURH (CARs)** | Registro de usuarios con concesión de aguas | Mensual | Solicitud institucional a cada CAR |
| **SUI (Superservicios)** | Consumo facturado por acueducto municipal | Mensual | [sui.gov.co](https://www.sui.gov.co) |
| **ENA (IDEAM)** | Balances hídricos nacionales y demanda sectorial | Quinquenal | [ideam.gov.co](http://www.ideam.gov.co) |
| **Medidores propios** | Lecturas directas de caudal in situ | Diaria | Sistema de telemetría / reportes internos |

**Variables adicionales útiles:**
- `precipitacion_mm`: Lluvia en la cuenca abastecedora (afecta la oferta disponible).
- `temperatura_c`: Afecta la demanda (meses calurosos = mayor consumo).
- `poblacion`: Para normalizar el consumo per cápita y comparar entre períodos.
- `ianc_pct`: Índice de Agua No Contabilizada (% de pérdidas del sistema).

> Colocar el archivo en `data/raw/` y ajustar la ruta. Serie típica: mensual por 5–15 años (60–180 observaciones).  
> Conectar con SIRH: disponible vía `load_ideam_dhime()` o exportación manual del portal.

In [ ]:
# df = load_csv("data/raw/pueea.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "consumo_agua": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

## 2. Validación y EDA

### Rangos físicos y alertas para consumo de agua

La función `validate()` aplica los rangos específicos para `pueea`:

- **Consumo de agua:** Siempre positivo (> 0 m³/mes). Un valor de 0 puede indicar error de medición, cierre temporal o problema de reporte.
- **Saltos bruscos:** Un incremento mayor al 50% respecto al mes anterior sin justificación (incendio, ampliación de planta, evento extraordinario) puede ser señal de fuga o captación no autorizada.
- **Caídas súbitas:** Un consumo repentinamente muy bajo puede indicar falla del medidor o corte de la fuente abastecedora.

**Problemas comunes en datos del SIRH:**
- **Subregistro:** Alta heterogeneidad en datos de caudal; pequeños usuarios frecuentemente sin medidores calibrados.
- **Retrasos en reporte:** Los usuarios deben reportar mensualmente pero el cargue efectivo al SIRH puede demorar 2–3 meses.
- **Unidades mixtas:** Verificar que todos los registros estén en m³/mes (algunos reportes se expresan en L/s o L/día).

> **ADR-002:** Consumos anómalos (picos o caídas súbitas) no se eliminan automáticamente. Son la señal más importante para detectar fugas, captaciones ilegales o eficiencia mejorada. El Isolation Forest (disponible en versiones futuras del módulo) puede ayudar a clasificar anomalías de forma automatizada.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_pueea.html",
       title="EDA — PUEEA", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

### Qué buscar en la visualización del consumo de agua

La serie mensual de consumo de agua combina tendencia, estacionalidad y anomalías. Al visualizar:

- **Tendencia de largo plazo:** ¿El consumo crece (ineficiencia), se mantiene estable o decrece (mejora en eficiencia)? Una tendencia decreciente estadísticamente significativa es evidencia de que el PUEEA está funcionando.
- **Estacionalidad:** El consumo de agua suele ser **débilmente estacional** para usuarios industriales o agroindustriales, pero más pronunciado en acueductos municipales donde la demanda sube en verano (temporada seca) y baja en invierno. Para consumo con estacionalidad débil, el modelo ETS multiplicativo es la opción predeterminada.
- **Línea base del PUEEA:** Visualizar el promedio del período de referencia (primeros 2 años del programa) como línea horizontal. La distancia entre el consumo actual y esta línea indica el avance hacia la meta del 15%.
- **Meses atípicos:** Picos o caídas que se separan visiblemente del patrón normal. Anotar en el gráfico con la causa conocida (obras, mantenimiento, eventos climáticos extremos).

In [ ]:
plot_series(df, "fecha", "consumo_agua", title="PUEEA — consumo_agua (m³)")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "consumo_agua", period="month")
plt.show()

## 3c. IANC — Perdidas de agua y deteccion de anomalias (Isolation Forest)

El **IANC (Indice de Agua No Contabilizada)** mide las perdidas del sistema de acueducto como porcentaje del volumen captado. Meta PUEAA: IANC <= 30% (Res. 943/2021 CRA).

```
IANC = (Volumen_captado - Volumen_facturado) / Volumen_captado * 100 (%)
Perdidas tecnicas: fugas en red, reboses en tanques
Perdidas comerciales: robos de agua, errores en medicion
```

**Isolation Forest** detecta lecturas de caudal anomalas (fugas subitas, captaciones ilegales, fallos de sensor).

In [ ]:
from sklearn.ensemble import IsolationForest

np.random.seed(66)
n = len(df)

# Volumen captado vs facturado (m3/mes)
vol_captado   = df['consumo_agua'].values * 1000  # proxy: consumo -> captacion
vol_facturado = vol_captado * np.clip(np.random.normal(0.72, 0.08, n), 0.3, 0.95)

# IANC mensual
ianc = (1 - vol_facturado / vol_captado) * 100
df['ianc_pct'] = ianc.round(1)

# Isolation Forest sobre caudal diario simulado
np.random.seed(7)
N_dias = 365
caudal_diario = np.random.normal(15, 3, N_dias)  # L/s
# Inyectar anomalias (fugas/captaciones ilegales)
idx_anom = np.random.choice(N_dias, 12, replace=False)
caudal_diario[idx_anom] += np.random.uniform(15, 40, 12)

iso_caudal = IsolationForest(contamination=0.05, random_state=42)
pred_caudal = iso_caudal.fit_predict(caudal_diario.reshape(-1, 1))
scores_caudal = iso_caudal.decision_function(caudal_diario.reshape(-1, 1))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Panel A: IANC en el tiempo
ax = axes[0]
colors_ianc = ['#e74c3c' if v > 30 else '#f1c40f' if v > 20 else '#27ae60'
               for v in df['ianc_pct']]
ax.bar(df['fecha'], df['ianc_pct'], color=colors_ianc, width=20, alpha=0.85)
ax.axhline(30, color='red', ls='--', lw=1.5, label='Meta IANC <= 30% (Res. 943/2021)')
ax.set_title('IANC — Indice Agua No Contabilizada (%)', fontweight='bold')
ax.set_ylabel('IANC (%)'); ax.legend(fontsize=7); ax.grid(axis='y', alpha=0.3)

# Panel B: Caudal diario + anomalias Isolation Forest
ax = axes[1]
t_dias = range(N_dias)
colors_if = ['#e74c3c' if p == -1 else '#2980b9' for p in pred_caudal]
ax.scatter(t_dias, caudal_diario, c=colors_if, s=8, alpha=0.7)
ax.set_title('Isolation Forest — Anomalias caudal diario\n(fugas, captaciones ilegales)', fontweight='bold')
ax.set_xlabel('Dia'); ax.set_ylabel('Caudal (L/s)')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#e74c3c',label='Anomalia'),
                   Patch(color='#2980b9',label='Normal')], fontsize=8)
ax.grid(alpha=0.3)

# Panel C: Tipo de perdidas
ax = axes[2]
perdidas_tecnicas   = np.clip(ianc * 0.65, 0, None).mean()
perdidas_comerciales= np.clip(ianc * 0.35, 0, None).mean()
ax.bar(['Tecnicas\n(fugas/reboses)', 'Comerciales\n(fraude/medicion)'],
       [perdidas_tecnicas, perdidas_comerciales],
       color=['#e74c3c', '#e67e22'], alpha=0.85, width=0.5)
ax.set_title('Composicion IANC promedio\n(PUEAA: meta reduccion quinquenal)', fontweight='bold')
ax.set_ylabel('IANC medio (%)'); ax.grid(axis='y', alpha=0.3)

plt.suptitle('PUEAA — IANC + Anomalias Isolation Forest (Ley 373/1997)',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show()

n_anom_if = (pred_caudal == -1).sum()
print(f'IANC promedio: {ianc.mean():.1f}% | Meses > 30%: {(ianc > 30).sum()}/{n}')
print(f'Anomalias caudal detectadas: {n_anom_if}/365 dias ({n_anom_if/365*100:.1f}%)')
print(f'Injected anomalias: {len(idx_anom)} — todas dentro de los {n_anom_if} detectados')

## 3b. Covariable ENSO (ONI)

### ENSO y consumo de agua: la conexión hidrológica

Para el PUEEA, el ENSO es relevante principalmente como modulador de la **oferta hídrica** disponible para los usuarios:

- **El Niño (ONI > +0.5°C):** Reduce precipitación → disminuye caudales de las fuentes abastecedoras → estrés hídrico → mayor presión sobre las concesiones → posible aumento del consumo de agua subterránea o racionamiento.
- **La Niña (ONI < −0.5°C):** Aumenta precipitación → mayor disponibilidad → menor presión sobre el sistema.

El lag de ENSO para `pueea` refleja el tiempo que tarda el ENSO en manifestarse en los caudales de las fuentes de abastecimiento, que varía según la cuenca. El valor de `config.ENSO_LAG_MESES` para esta línea es el apropiado según la literatura colombiana (IDEAM, ENA).

**Usos analíticos del ONI para PUEEA:**
1. Como covariable en SARIMAX para mejorar el pronóstico de demanda en años de evento climático.
2. Para contextualizar anomalías de consumo: un pico en año Niño puede deberse a reducción de la oferta (no a incremento de la demanda real).
3. Para planificar el PUEAA en escenarios de cambio climático: ¿cómo se comporta el sistema en un año Niño intenso?

In [ ]:
# --- Covariable ENSO (lag específico para PUEEA) ---
oni = load_oni()  # Descarga ONI desde NOAA
df = enso_lagged(df, oni, date_col="fecha", linea_tematica="pueea")
print("Columnas ENSO agregadas:", [c for c in df.columns if "oni" in c or "enso" in c])

## 4. Estadística descriptiva

### Indicadores clave para el PUEEA

| Estadístico | Interpretación para gestión del agua |
|---|---|
| **Media mensual** | Consumo típico; base para calcular la meta de reducción del 15% |
| **Percentil 95** | Consumo en picos extremos; determina la capacidad de reserva requerida |
| **Coeficiente de variación (CV)** | CV > 20% indica variabilidad significativa → explorar causas (fugas estacionales, procesos irregulares) |
| **Tendencia (slope Mann-Kendall)** | Positiva = consumo creciente (ineficiencia); negativa = mejora en eficiencia |
| **Consumo per cápita** | Si se dispone de datos de población, normalizar para comparar con benchmark sectorial |

**Benchmarks de referencia (IDEAM / ENA 2022):**

| Sector usuario | Consumo típico |
|---|---|
| Acueducto municipal pequeño (<10.000 hab.) | 80–150 L/hab/día |
| Acueducto municipal grande (>100.000 hab.) | 120–200 L/hab/día |
| Meta eficiencia OMS (agua potable) | 50–100 L/hab/día |
| IANC aceptable (pérdidas) | < 30% |

**Eficiencia hídrica — cálculo de la meta del PUEEA:**
```python
# Línea base: promedio de los primeros 24 meses del programa
linea_base = df["consumo_agua"][:24].mean()
meta_reduccion = linea_base * 0.85  # -15% de la línea base
print(f"Línea base PUEEA: {linea_base:,.0f} m³/mes")
print(f"Meta PUEEA (−15%): {meta_reduccion:,.0f} m³/mes")
print(f"Consumo actual: {df['consumo_agua'].iloc[-12:].mean():,.0f} m³/mes")
```

In [ ]:
summarize(df)

## 5. Inferencial

### Estacionariedad: importancia para el pronóstico de consumo

El consumo de agua puede tener o no tendencia dependiendo de la fase del PUEEA:

- **Fase inicial (primeros 2–3 años):** Consumo estable o con leve tendencia creciente por expansión productiva → probable no estacionariedad (ADF: p > 0.05).
- **Fase de eficiencia (años 3–8):** Consumo con tendencia decreciente por medidas del PUEAA → no estacionariedad con tendencia negativa.
- **Fase madura:** Consumo estacionario alrededor de un nivel de eficiencia → ADF rechaza raíz unitaria.

**ADR-004 — Prueba dual ADF + KPSS:**
- Si la serie es **no estacionaria**: diferencias el consumo (d=1) antes de SARIMA, o usar directamente Prophet/XGBoost que no requieren estacionariedad.
- Para SARIMAX con covariables climáticas, verificar también la estacionariedad del regresor ONI (generalmente estacionario).

### Mann-Kendall: ¿el PUEEA está funcionando?

La tendencia Mann-Kendall sobre el consumo mensual responde directamente la pregunta central del PUEEA:

- **Tendencia decreciente significativa (p < 0.05, slope < 0):** El PUEAA produce reducción sostenida del consumo. Evidencia positiva para el informe a la CAR.
- **Sin tendencia (p ≥ 0.05):** El consumo es estable. Puede ser buena noticia si ya se alcanzó la meta, o preocupante si no se ha iniciado la reducción.
- **Tendencia creciente (slope > 0):** El PUEAA no está siendo efectivo. Requiere revisión de metas y medidas de eficiencia.

In [ ]:
ts = df.set_index("fecha")["consumo_agua"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5b. Análisis de excedencias normativas

### Excedencias del PUEEA: consumo por encima de la línea base

Para el PUEEA, el análisis de excedencias evalúa cuántos meses el consumo **superó la línea base del programa** (o la meta de reducción). Esto alimenta directamente los informes de seguimiento a la CAR:

| Umbral | Valor | Fuente | Interpretación |
|---|---|---|---|
| **Línea base PUEEA** | Media de primeros 24 meses | Resolución 1257/2018 | Referencia de partida |
| **Meta PUEEA (−15%)** | Línea base × 0.85 | IDEAM / Meta sectorial | Nivel objetivo de eficiencia |
| **IANC aceptable** | < 30% de pérdidas | ENA 2022 / IDEAM | Umbral operativo de eficiencia de red |

**Cálculo de excedencias respecto a línea base:**
```python
linea_base = df["consumo_agua"][:24].mean()
# Excedencias: meses con consumo > línea base (regresión respecto a la situación inicial)
excedencias = df[df["consumo_agua"] > linea_base]
print(f"Meses con consumo > línea base: {len(excedencias)} / {len(df)} ({len(excedencias)/len(df)*100:.1f}%)")

# Excedencias respecto a meta -15%
meta = linea_base * 0.85
meses_fuera_meta = df[df["consumo_agua"] > meta]
print(f"Meses sin alcanzar la meta de −15%: {len(meses_fuera_meta)} / {len(df)}")
```

**Para integrar con `exceedance_report()`:** Usar el valor de la línea base como threshold manual. La tabla resultante se puede incluir directamente en el informe anual del PUEAA a la CAR.

In [ ]:
rep = exceedance_report(df["consumo_agua"], variable="consumo_agua")
if rep.empty:
    print("Sin normas colombianas registradas para 'consumo_agua'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento

### Tratamiento de datos de consumo de agua

**Faltantes comunes y cómo manejarlos:**

| Causa del faltante | Estrategia |
|---|---|
| Falla del medidor (1–2 meses) | Interpolación lineal o promedio climatológico del mismo mes en otros años |
| Período sin operación (mantenimiento) | No imputar; marcar como dato no disponible y documentar |
| Retraso en reporte al SIRH | Completar con datos de medidores propios si están disponibles |
| Mes de cierre de planta / parada técnica | Imputar con 0 solo si se confirma que no hubo captación |

**Conversión de unidades (frecuente en datos del SIRH):**
```python
# L/s → m³/mes (aproximación con 30 días)
df["consumo_agua_m3"] = df["consumo_l_s"] * 86400 * 30 / 1000

# L/día → m³/mes
df["consumo_agua_m3"] = df["consumo_l_dia"] * 30 / 1000
```

> **ADR-002:** Los picos de consumo (p.ej., meses con pluviosidad muy baja durante El Niño donde se capta más agua subterránea) no se eliminan. Son la señal que justifica la actualización del PUEEA para escenarios de variabilidad climática.

In [ ]:
df_clean = impute(df.copy(), cols=["consumo_agua"], method="linear")
print(f"Faltantes antes: {df["consumo_agua"].isna().sum()} | "
      f"después: {df_clean["consumo_agua"].isna().sum()}")

## 7. Modelos predictivos

### Selección de modelo para consumo de agua en el PUEEA

El pronóstico de consumo de agua permite planificar la disponibilidad del recurso y verificar si las medidas de eficiencia del PUEAA alcanzan las metas proyectadas.

| Modelo | Ventajas para consumo de agua | Cuándo preferirlo |
|---|---|---|
| **SARIMA** | Captura estacionalidad mensual (S=12) sin covariables | Consumo con estacionalidad regular, sin tendencia fuerte |
| **SARIMAX** | Incorpora precipitación, temperatura u ONI como regresores | Demanda sensible al clima (acueductos rurales, riego) |
| **Prophet** | Maneja tendencias complejas, estacionalidad múltiple y eventos atípicos | Consumo con cambios de tendencia por implementación del PUEAA |
| **XGBoost** | Captura relaciones no lineales; útil con múltiples covariables | Serie larga con covariables disponibles (ONI, temperatura, IANC) |

**Estacionalidad débil — ETS multiplicativo:**
Para consumo industrial con estacionalidad suave, el modelo ETS (Error-Trend-Seasonal) con componente estacional multiplicativo es una alternativa robusta y de menor complejidad que SARIMA.

**Métricas de evaluación para PUEEA (ADR-003):**
- **RMSE:** Métrica principal — penaliza desvíos grandes respecto a la meta.
- **MAE:** Métrica secundaria — más robusta a valores atípicos (meses de mantenimiento).
- **sMAPE:** Permite comparar el error relativo entre usuarios o instalaciones de diferente tamaño.
- **No usar RMSLE** para consumo de agua (ADR-003: riesgo de sesgar la evaluación hacia errores en valores bajos).

**Horizonte de pronóstico:** 6 meses es el estándar operativo para la planificación hídrica. El informe anual del PUEAA a la CAR puede usar proyecciones a 12 meses.

In [ ]:
ts = df_clean.set_index("fecha")["consumo_agua"]

models = {
    "SARIMA":       get_model("sarima", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "SARIMAX":      get_model("sarimax", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "Prophet":      get_model("prophet"),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 8. Conclusiones

### Plantilla de conclusiones (completar con datos reales)

**Tendencia de consumo:**
- Resultado Mann-Kendall: `[creciente / decreciente / sin tendencia]`, slope = `X` m³/mes por año (p = `Y`).
- Interpretación: El PUEAA `[está siendo efectivo / no está produciendo reducciones / necesita revisión]`.

**Eficiencia hídrica respecto a la meta:**
- Línea base PUEAA (promedio de primeros 24 meses): `X` m³/mes.
- Meta de reducción (−15%): `Y` m³/mes.
- Consumo promedio en últimos 12 meses: `Z` m³/mes.
- Estado: `[Meta alcanzada / En progreso — X% de reducción lograda / Sin avance]`.

**Excedencias:**
- Meses con consumo > línea base: `N` de `T` meses (`X%` del período).
- Meses sin alcanzar la meta de −15%: `M` de `T` meses.

**Anomalías detectadas:**
- Meses atípicos identificados visualmente: `[lista de fechas y posibles causas]`.

**Mejor modelo predictivo:**
- Modelo seleccionado: `[nombre]` con RMSE = `X` m³, MAE = `Y` m³, sMAPE = `Z%`.
- Proyección a 6 meses: `[rango esperado de consumo]` m³/mes.

### Normativa y referencias

- **Ley 373/1997:** Obligatoriedad del PUEEA para usuarios con concesión.
- **Resolución 1257/2018:** Estructura del PUEAA y PUEAA Simplificado.
- **Meta IDEAM:** Reducción del 15% del consumo respecto a la línea base.
- **IANC < 30%:** Meta de pérdidas técnicas en red de acueducto (ENA 2022).
- Registrar decisiones metodológicas en [`docs/decisiones.md`](../../docs/decisiones.md).

## 9. Cómo adaptar a datos reales

### Paso 1: Obtener datos de consumo de agua

```python
# Opción A: SIRH / RURH (concesiones reportadas a la CAR)
# Exportar desde el portal del SIRH o solicitar a la CAR
df = load_csv("data/raw/consumo_agua_concesion_XYZ.csv", date_col="fecha")
# Columnas esperadas: fecha, consumo_agua (m³/mes), fuente_captacion, usuario

# Opción B: SUI (Superservicios) para acueductos municipales
# URL: https://www.sui.gov.co → Reportes → Acueducto → Consumo facturado
df = load_csv("data/raw/consumo_sui_municipio.csv", date_col="fecha")

# Opción C: Medidores propios (telemetría o lecturas manuales)
df = load_csv("data/raw/lecturas_medidor_planta.csv", date_col="fecha")
```

### Paso 2: Calcular la línea base y la meta del PUEEA

```python
# Definir los primeros 24 meses como período de referencia
MESES_LINEA_BASE = 24
META_REDUCCION = 0.15  # 15% según IDEAM

linea_base = df["consumo_agua"].iloc[:MESES_LINEA_BASE].mean()
meta_pueea = linea_base * (1 - META_REDUCCION)

print(f"Línea base: {linea_base:,.1f} m³/mes")
print(f"Meta PUEEA (-{META_REDUCCION*100:.0f}%): {meta_pueea:,.1f} m³/mes")

# Agregar al DataFrame para visualización
df["linea_base"] = linea_base
df["meta_pueea"] = meta_pueea
```

### Paso 3: Análisis de excedencias respecto a la meta

```python
from estadistica_ambiental.inference.intervals import exceedance_report

# Meses donde el consumo superó la línea base (sin mejora)
rep_lb = exceedance_report(df["consumo_agua"], variable="consumo_agua",
                           threshold=linea_base)
print(f"Meses por encima de línea base: {rep_lb['n_exceedances'].sum()}")

# Meses donde el consumo superó la meta de eficiencia
rep_meta = exceedance_report(df["consumo_agua"], variable="consumo_agua",
                              threshold=meta_pueea)
print(f"Meses sin alcanzar la meta (-15%): {rep_meta['n_exceedances'].sum()}")
```

### Paso 4: Convertir unidades si es necesario

```python
# L/s → m³/mes
if "consumo_l_s" in df.columns:
    df["consumo_agua"] = df["consumo_l_s"] * 86400 * 30 / 1000

# L/día → m³/mes
if "consumo_l_dia" in df.columns:
    df["consumo_agua"] = df["consumo_l_dia"] * 30 / 1000
```

### Paso 5: Detección de anomalías de consumo

```python
# Detección simple: consumo > media + 3 desviaciones estándar
umbral_anomalia = df["consumo_agua"].mean() + 3 * df["consumo_agua"].std()
anomalias = df[df["consumo_agua"] > umbral_anomalia]
print(f"Meses con consumo anómalo (>media+3σ): {len(anomalias)}")
print(anomalias[["fecha", "consumo_agua"]])
# Investigar cada mes anómalo con registros operativos del usuario
```

### Paso 6: Generar informe de cumplimiento PUEEA para la CAR

```python
from estadistica_ambiental.reporting.compliance_report import compliance_report

compliance_report(
    df,
    variables=["consumo_agua"],
    linea_tematica="pueea",
    output="data/output/reports/cumplimiento_pueea.html"
)
# Abrir el HTML para el informe de cumplimiento listo para presentar a la CAR
```

### Paso 7: Consideraciones especiales

- **Series cortas (<5 años):** Preferir Prophet o regresión de tendencia sobre SARIMA; el SARIMA necesita al menos 3–5 ciclos estacionales completos.
- **Cambio de proceso productivo:** Si el usuario amplió su capacidad o cambió su proceso, documentar la fecha como punto de ruptura y re-calcular la línea base desde ese momento.
- **Reporte a la CAR:** Los resultados del análisis Mann-Kendall y las excedencias son la base del informe anual del PUEAA. Exportar la tabla de excedencias a Excel para adjuntar al informe formal.
- Registrar decisiones metodológicas en [`docs/decisiones.md`](../../docs/decisiones.md).